In [ ]:
import anndata as ad
import os
import re
import numpy as np
import squidpy as sq
import scanpy as sc
import harmonypy as hm
import umap

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### Load datasets with your input path directories

In [ ]:
# Load nuclear protein intensity matrices of the two brain sections.
# Brain 1 (~178927 cells, 18 markers) and Brain 2 (~194890 cells, 15 markers)
# are pre-saved h5ad snapshots produced by the upstream segmentation pipeline.
adata_1st_brain = ad.read_h5ad('input/adata_18_nuclear_46_prot_brain1.h5ad')
adata_2nd_brain = ad.read_h5ad('input/brain2_nuclear_int_18prot.h5ad')

# Prefix obs_names so we can disambiguate same-numbered cells across brains
# AFTER concat (this prevents the row-order alignment bugs we hit earlier).
adata_1st_brain.obs_names = 'b1_' + adata_1st_brain.obs_names.astype(str).str.strip()
adata_2nd_brain.obs_names = 'b2_' + adata_2nd_brain.obs_names.astype(str).str.strip()

common_vars = adata_1st_brain.var_names.intersection(adata_2nd_brain.var_names)
adata_1 = adata_1st_brain[:, common_vars].copy()
adata_2 = adata_2nd_brain[:, common_vars].copy()

In [ ]:
# Load single-cell RNA cycleHCR data from RNA spot-to-cell assignment, BOTH brains.
rna_b1_df = pd.read_csv("input/cell_by_transcript_gene_name_matrix1.csv", index_col=0)
rna_b2_df = pd.read_csv("input/cell_by_transcript_gene_name_matrix2.csv", index_col=0)

In [ ]:
adata_both = ad.concat([adata_1, adata_2],
                      join="outer",     # keep all features
                      label="dataset",  # new column in .obs
                      keys=["1st_brain", "2nd_brain"])  

In [ ]:
common_genes = adata_1.var_names.intersection(adata_2.var_names)
len(common_genes)

## Make integrated UMAP of two brain sections using Harmony

In [ ]:
adata_both.layers["counts"] = adata_both.X.copy()
sc.pp.normalize_total(adata_both, inplace=True)
sc.pp.log1p(adata_both)

In [ ]:
sc.tl.pca(adata_both, svd_solver="arpack")

In [ ]:
ho = hm.run_harmony(adata_both.obsm['X_pca'], adata_both.obs, 'dataset', theta=6, lamb = 0.3, sigma = 0.02, nclust=100, max_iter_harmony = 20, random_state=42)   # 
# theta (diversity penalty) Default: 2 Larger values (e.g. 6, 8, 12) are used when default cannot resolve batch effect.
# Harmony run takes ~30 mins for our dataset

# save Harmony-corrected embeddings
adata_both.obsm['X_pca_harmony'] = ho.Z_corr.T

In [ ]:
X_pca_1 = adata_both[adata_both.obs["dataset"]=="1st_brain"].obsm["X_pca_harmony"]
X_pca_2 = adata_both[adata_both.obs["dataset"]=="2nd_brain"].obsm["X_pca_harmony"]

In [ ]:
reducer = umap.UMAP(n_neighbors=5, min_dist=0.1, n_components=2, random_state=42)
reducer.fit(X_pca_2)
X_umap_1 = reducer.transform(X_pca_1)
X_umap_2 = reducer.transform(X_pca_2)

In [ ]:
adata_1.obsm["X_umap"] = X_umap_1
adata_2.obsm["X_umap"] = X_umap_2

adata_plot = ad.concat(
    [adata_1, adata_2],
    join="outer",
    label="dataset",
    keys=["1st brain section", "2nd brain section"],
    merge="unique"
)

adata_plot.obsm['X_pca_harmony'] = np.vstack([X_pca_1, X_pca_2])
adata_plot.obsm["X_umap"] = np.vstack([X_umap_1, X_umap_2])

# 7) Plot
sc.pl.umap(adata_plot, color=["dataset"])

In [ ]:
# Save matrix to skip rerunning Harmony everytime

# adata_plot.write('adata_plot_harmony_ave_int_2_brains.h5ad')   

In [ ]:
# adata_plot= ad.read('adata_plot_harmony_ave_int_2_brains.h5ad')

In [ ]:
sc.set_figure_params(figsize=(15, 15))
sc.pl.umap(adata_plot, color=["dataset"], size=5, alpha=0.9)

# plt.savefig("harmony_umap.eps", format="eps", bbox_inches="tight")
plt.close()

In [ ]:
print("neighbors")
sc.pp.neighbors(adata_plot, n_neighbors=10, random_state=42, use_rep='X_pca_harmony')

In [ ]:
print("Leiden")
resolution = 0.8       # 1 -- 40 clusters
sc.tl.leiden(adata_plot, resolution=resolution, random_state=38)

In [ ]:
sc.set_figure_params(figsize=(15, 15))
sc.pl.umap(adata_plot, color=["leiden"], size=8)

In [ ]:
ad_1st = adata_plot[adata_plot.obs.dataset == '1st brain section']
ad_2nd = adata_plot[adata_plot.obs.dataset == '2nd brain section']

In [ ]:
sc.pl.umap(ad_1st, color=["leiden"],size=7)

In [ ]:
sq.pl.spatial_scatter(ad_1st, shape=None, color="leiden", size=6, library_id="one")

In [ ]:
sq.pl.spatial_scatter(ad_2nd, shape=None, color="leiden", size=6, library_id="one")

In [ ]:
# Save clustering results.

adata_plot.write('harmony_ave_int_2_brains_33_clusters.h5ad')   
ad_1st.write('harmony_ave_int_Brain_1_32_clusters.h5ad')

In [ ]:
# Load clustering result
adata_plot = ad.read_h5ad('harmony_ave_int_2_brains.h5ad')
ad_1st = ad.read_h5ad('harmony_ave_int_Brain_1.h5ad') 

In [ ]:
ad_2nd = adata_plot[adata_plot.obs.dataset == '2nd brain section']

In [ ]:
import matplotlib.patheffects as PathEffects

sc.set_figure_params(figsize=(15, 15))

# Plot UMAP with on-data labels
sc.pl.umap(ad_1st, color="leiden", legend_loc="on data", size=7, show=False)

ax = plt.gca()

ax.set_frame_on(False)   # removes the box
ax.set_xticks([])        # removes x ticks
ax.set_yticks([])        # removes y ticks
ax.set_xlabel("")        # removes x-axis title
ax.set_ylabel("")        # removes y-axis title
ax.set_title("")  

for txt in ax.texts:
    txt.set_fontsize(20)
    txt.set_fontname("Arial")
    txt.set_path_effects([
        PathEffects.withStroke(linewidth=1, foreground='white')  # white outline
    ])

for coll in ax.collections:
    coll.set_rasterized(True)

# plt.savefig("umap_2_brains.pdf", dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()

In [ ]:
# rotate spatial coordinates for plotting the brain section

# 1. Backup original coordinates
original_coords = ad_1st.obsm["spatial"].copy()

# 2. Rotate coordinates
theta = np.radians(-137)
rotation_matrix = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])
rotated_coords = original_coords @ rotation_matrix.T
ad_1st.obsm["spatial_rotated"] = rotated_coords

In [ ]:
# Orient another brain section
original_coords_2 = ad_2nd.obsm["spatial"].copy()

theta = np.radians(53)
rotation_matrix = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])
rotated_coords_2 = original_coords_2 @ rotation_matrix.T
ad_2nd.obsm["spatial_rotated"] = rotated_coords_2

In [ ]:
coords1 = ad_1st.obsm['spatial_rotated']
coords2 = ad_2nd.obsm['spatial_rotated']

leiden_categories = adata_plot.obs['leiden'].cat.categories
leiden_colors = adata_plot.uns['leiden_colors']
cluster_colors = dict(zip(leiden_categories, leiden_colors))

clusters = [str(i) for i in range(0, 33)]
n_clusters = len(clusters)
bg_color = "#EEEEEE"

# Two cluster pairs per row (4 columns: B1 B2 | B1 B2)
ncols = 4
clusters_per_row = ncols // 2
nrows = (n_clusters + clusters_per_row - 1) // clusters_per_row

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.reshape(nrows, ncols)

# Track which axes we actually use
used_axes = set()

for i, cluster in enumerate(clusters):
  row, pair_col = divmod(i, clusters_per_row)
  col_b1 = pair_col * 2
  col_b2 = col_b1 + 1

  for ax, ad_plot, coords, brain_label in [
      (axes[row, col_b1], ad_1st, coords1, '1st'),
      (axes[row, col_b2], ad_2nd, coords2, '2nd'),
  ]:
      used_axes.add((row, ax.get_subplotspec().colspan.start))

      ax.scatter(coords[:, 0], coords[:, 1],
                 c=bg_color, s=1, rasterized=True)

      mask = (ad_plot.obs['leiden'] == cluster).values
      n = int(mask.sum())
      if n > 0:
          ax.scatter(coords[mask, 0], coords[mask, 1],
                     c=cluster_colors.get(cluster, 'red'),
                     s=2, rasterized=True)

      ax.set_title(f"{brain_label} — cluster {cluster} (n={n})", fontsize=11)
      ax.set_aspect('equal')
      ax.invert_yaxis()
      ax.axis('off')

# Hide any axes that weren't used (e.g. trailing panels in the last row)
for r in range(nrows):
  for c in range(ncols):
      if (r, c) not in used_axes:
          axes[r, c].set_visible(False)

plt.tight_layout()
# plt.savefig('clusters_side_by_side.pdf', dpi=200, bbox_inches='tight')
plt.show()

## RNA feature of nuclear-protein-defined clusters

In [ ]:
# Customize thresholds for cell cluster analysis

MIN_RATIO = 0.20
# Filter: a Harmony-integrated cluster is included only if min(n_B1, n_B2) / max(n_B1, n_B2) >=
# MIN_RATIO. This excludes clusters whose RNA-matched cell count is heavily
# imbalanced across brains (e.g., 5000 vs 30 cells), where the marker
# ranking is dominated by one brain and direct comparison isn't meaningful.



MIN_N = 500
# For every cluster that survived BOTH the cross-brain ratio filter
# (previous cell) AND has > MIN_N RNA-matched cells in each brain, draw a
# row showing: cluster #, spatial distribution in brain 1, spatial
# distribution in brain 2, and the shared top-5 genes.

In [ ]:
# Restrict to genes detected in both brains so every comparison operates on
# the same feature set.
common_genes = rna_b1_df.columns.intersection(rna_b2_df.columns)
rna_b1_df = rna_b1_df[common_genes]
rna_b2_df = rna_b2_df[common_genes]
print(f"Brain 1 RNA: {rna_b1_df.shape[0]} cells x {rna_b1_df.shape[1]} genes")
print(f"Brain 2 RNA: {rna_b2_df.shape[0]} cells x {rna_b2_df.shape[1]} genes")
print(f"Common genes: {len(common_genes)}")


In [ ]:
# Build per-brain RNA AnnDatas, carrying the joint Leiden labels from the
# Harmony-integrated protein clustering. Only cells with both protein
# (`ad_*` membership) and RNA (`rna_*_df` index) survive the intersection.

def _make_rna_adata(rna_df, ad_brain):
    rna_df = rna_df.copy()
    rna_df.index = rna_df.index.astype(str).str.strip()
    common_cells = ad_brain.obs_names.intersection(rna_df.index)
    rna_df = rna_df.loc[common_cells]
    out = ad.AnnData(
        X=rna_df.to_numpy(),
        obs=ad_brain.obs.loc[common_cells].copy(),
        var=pd.DataFrame(index=rna_df.columns.astype(str)),
    )
    out.obs_names = common_cells
    out.var_names = rna_df.columns.astype(str)
    for k in ad_brain.obsm.keys():
        out.obsm[k] = ad_brain.obsm[k][ad_brain.obs_names.get_indexer(common_cells)]
    out.uns = ad_brain.uns.copy()
    return out


adata_rna_b1 = _make_rna_adata(rna_b1_df, ad_1st)
adata_rna_b2 = _make_rna_adata(rna_b2_df, ad_2nd)

# Backwards-compat alias for any downstream cell that still references
# `adata_rna` (the original single-brain object).
adata_rna = adata_rna_b1

print(f"Brain 1: {adata_rna_b1.n_obs} cells matched to RNA")
print(f"Brain 2: {adata_rna_b2.n_obs} cells matched to RNA")


In [ ]:
sc.pp.normalize_total(adata_rna_b1, inplace=True)
sc.pp.log1p(adata_rna_b1)
sc.pp.normalize_total(adata_rna_b2, inplace=True)
sc.pp.log1p(adata_rna_b2)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

# rank_genes_groups raises ValueError if any selected group contains < 2
# cells. Joint-Leiden cluster IDs are shared across brains, but small or
# empty clusters can appear in one brain after the RNA-cell intersection
# (e.g., cluster '31' has only 1 RNA-matched cell in brain 2). We restrict
# the test to clusters with >= 2 cells in each brain.
def _multi_cell_groups(ad_rna, min_cells=2):
    sizes = ad_rna.obs['leiden'].astype(str).value_counts()
    keep = sizes[sizes >= min_cells].index.tolist()
    return sorted(keep, key=lambda x: int(x) if x.isdigit() else 10**9)


groups_b1 = _multi_cell_groups(adata_rna_b1)
groups_b2 = _multi_cell_groups(adata_rna_b2)

dropped_b1 = sorted(set(adata_rna_b1.obs['leiden'].astype(str).unique()) - set(groups_b1))
dropped_b2 = sorted(set(adata_rna_b2.obs['leiden'].astype(str).unique()) - set(groups_b2))
if dropped_b1: print(f"B1: skipping singleton/empty clusters in rank_genes: {dropped_b1}")
if dropped_b2: print(f"B2: skipping singleton/empty clusters in rank_genes: {dropped_b2}")

sc.tl.rank_genes_groups(adata_rna_b1, groupby="leiden", method="wilcoxon", groups=groups_b1)
sc.tl.rank_genes_groups(adata_rna_b2, groupby="leiden", method="wilcoxon", groups=groups_b2)


In [ ]:
def _wide_marker_df(ad_):
    res = ad_.uns['rank_genes_groups']
    groups = res['names'].dtype.names
    return pd.DataFrame({
        f'{g}_{k[0]}': res[k][g]
        for g in groups
        for k in ['names', 'logfoldchanges', 'scores', 'pvals']
    })

dat_b1 = _wide_marker_df(adata_rna_b1)
dat_b2 = _wide_marker_df(adata_rna_b2)
# dat_b1.to_csv("rna_marker_genes_brain1.csv")
# dat_b2.to_csv("rna_marker_genes_brain2.csv")


In [ ]:
print("Brain 1 top markers per cluster:")
sc.pl.rank_genes_groups(adata_rna_b1, n_genes=15, fontsize=40)
print("Brain 2 top markers per cluster:")
sc.pl.rank_genes_groups(adata_rna_b2, n_genes=15, fontsize=40)


In [ ]:
# Top 5 marker genes of each cluster, side by side for the two brains.

def _top_n_per_cluster(ad_, n=5):
    res = ad_.uns['rank_genes_groups']
    return {g: list(res['names'][g][:n]) for g in res['names'].dtype.names}


top5_b1 = _top_n_per_cluster(adata_rna_b1, 5)
top5_b2 = _top_n_per_cluster(adata_rna_b2, 5)

sizes_b1 = adata_rna_b1.obs['leiden'].astype(str).value_counts()
sizes_b2 = adata_rna_b2.obs['leiden'].astype(str).value_counts()

clusters = sorted(set(top5_b1) | set(top5_b2),
                  key=lambda x: int(x) if x.isdigit() else 10**9)

rows, skipped = [], []
for c in clusters:
    n1, n2 = int(sizes_b1.get(c, 0)), int(sizes_b2.get(c, 0))
    hi, lo = max(n1, n2), min(n1, n2)
    ratio = (lo / hi) if hi > 0 else 0.0
    if ratio < MIN_RATIO:
        skipped.append((c, n1, n2, ratio))
        continue
    g1 = top5_b1.get(c, [])
    g2 = top5_b2.get(c, [])
    shared = sorted(set(g1) & set(g2))
    rows.append({
        'cluster': c,
        'n_b1': n1,
        'n_b2': n2,
        'ratio': round(ratio, 3),
        'B1_top5': ', '.join(g1) if g1 else '-',
        'B2_top5': ', '.join(g2) if g2 else '-',
        'shared':   ', '.join(shared),
        'n_shared': len(shared),
    })

top5_df = pd.DataFrame(rows)
print(top5_df.to_string(index=False))
print()
if skipped:
    print(f"Skipped {len(skipped)} cluster(s) with cross-brain ratio < {MIN_RATIO}:")
    for c, n1, n2, ratio in skipped:
        print(f"  cluster {c}: B1 n={n1}, B2 n={n2}, ratio={ratio:.3f}")
top5_df.to_csv('rna_top5_markers_b1_vs_b2.csv', index=False)


In [ ]:
bg_color = "#EEEEEE"

passed = top5_df[(top5_df['n_b1'] > MIN_N) & (top5_df['n_b2'] > MIN_N)]
filtered_clusters = passed['cluster'].tolist()
shared_lookup = dict(zip(passed['cluster'], passed['shared']))

dropped_low_n = top5_df[~((top5_df['n_b1'] > MIN_N) & (top5_df['n_b2'] > MIN_N))]
if len(dropped_low_n):
    print(f"Skipping {len(dropped_low_n)} cluster(s) with <= {MIN_N} cells in either brain:")
    print(dropped_low_n[['cluster', 'n_b1', 'n_b2']].to_string(index=False))
    print()

if not filtered_clusters:
    raise RuntimeError(f"No clusters passed (ratio >= 0.20 AND n > {MIN_N} in both brains).")

xy_b1 = ad_1st.obsm['spatial_rotated']
xy_b2 = ad_2nd.obsm['spatial_rotated']
labels_b1 = ad_1st.obs['leiden'].astype(str).values
labels_b2 = ad_2nd.obs['leiden'].astype(str).values

# Match the joint Leiden palette already used elsewhere in the notebook so a
# given cluster keeps the same color across figures.
leiden_categories = list(adata_plot.obs['leiden'].cat.categories)
leiden_colors     = list(adata_plot.uns['leiden_colors'])
cluster_color     = dict(zip(leiden_categories, leiden_colors))

n = len(filtered_clusters)
fig, axes = plt.subplots(
    n, 4,
    figsize=(18, 3.2 * n),
    gridspec_kw={'width_ratios': [0.4, 2, 2, 2]},
)
if n == 1:
    axes = axes.reshape(1, -1)

for row, cluster in enumerate(filtered_clusters):
    color = cluster_color.get(cluster, 'tab:red')
    shared_str = shared_lookup.get(cluster, '') or '(no shared)'

    # Col 0 -- cluster number.
    ax = axes[row, 0]
    ax.text(0.5, 0.5, cluster, fontsize=26, fontweight='bold',
            ha='center', va='center')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_frame_on(False)
    if row == 0:
        ax.set_title('Cluster', fontsize=26)

    # Col 1 -- brain 1 spatial.
    ax = axes[row, 1]
    mask = (labels_b1 == cluster)
    ax.scatter(xy_b1[~mask, 0], xy_b1[~mask, 1], s=1, c=bg_color, rasterized=True)
    ax.scatter(xy_b1[ mask, 0], xy_b1[ mask, 1], s=2, c=[color], rasterized=True)
    ax.set_aspect('equal'); ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])
    if row == 0:
        ax.set_title('Brain 1', fontsize=26)

    # Col 2 -- brain 2 spatial.
    ax = axes[row, 2]
    mask = (labels_b2 == cluster)
    ax.scatter(xy_b2[~mask, 0], xy_b2[~mask, 1], s=1, c=bg_color, rasterized=True)
    ax.scatter(xy_b2[ mask, 0], xy_b2[ mask, 1], s=2, c=[color], rasterized=True)
    ax.set_aspect('equal'); ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])
    if row == 0:
        ax.set_title('Brain 2', fontsize=26)

    # Col 3 -- shared top-5 genes text.
    ax = axes[row, 3]
    ax.text(0.02, 0.5, shared_str, fontsize=26, ha='left', va='center', wrap=True)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_frame_on(False)
    if row == 0:
        ax.set_title('Shared marker genes', fontsize=26)

plt.tight_layout()
plt.savefig('cluster_spatial_b1_b2.pdf', dpi=200, bbox_inches='tight')
plt.show()
